# 第 02 步：用复权收盘价计算简单日收益率

<!-- SNAPSHOT_NOTE -->

## 数据准备

本页记录的是完整快照上的实际运行。从 GitHub 克隆源码后，先在仓库根目录运行 `python publication/restore_snapshot.py`，恢复同版本的 `data/` 与 `runs/`，再执行本步。[完整数据与大型结果索引](../../publication/ASSET_INDEX.md)保留全部路径和哈希；本页图表与 Notebook 已保存输出可直接阅读。

## 目标、前置与运行方式

上一节完成了逐行读取，本步进一步解释同一只股票相邻两个模拟交易日之间的价格变化。关注点从“今天涨了多少元”转为可以跨价格尺度比较的收益率。研究对象仍是 `SYN001`，输入仍是 `runs/first_demo_20261107/input/prices.csv`，在这个步骤不更换股票或引入真实数据。

完成 [第 01 步](../01_read_prices/README.md) 的读取、筛选与排序后，在仓库根目录执行：

```powershell
.\.venv\Scripts\python.exe examples/02_daily_returns.py
```

本脚本只在内存中新增派生列并打印结果，不会把收益覆盖进原始行情。代码中保留原始价格列，方便把公式与输入逐项对照。

## 简单日收益率的定义

用 `P_t` 表示当前模拟交易日的复权收盘价，用 `P_(t-1)` 表示上一个模拟交易日的复权收盘价。简单日收益率定义为：

$$
r_t=\frac{P_t-P_{t-1}}{P_{t-1}}=\frac{P_t}{P_{t-1}}-1.
$$

这个公式的分母是上一期价格，衡量的是原有一单位价值经过这段价格变化后的增长比例。它不同于 `close-open` 的绝对价差，也不同于 `close/open-1` 的日内收益。后一种只覆盖当天开盘到收盘，遗漏前一收盘到当天开盘的隔夜变化。

本节计算的是复权价格序列上的价格变动代理，尚未建立完整的真实股票账户。计算中还没有判断当时是否持仓、能否成交、佣金是多少，也没有把收益乘上真实股数。因此，一列日收益不能直接解释为已经实现的组合损益。

## 先对齐独立日历

```python
calendar = pd.read_csv(
    data_path.with_name("calendar.csv"), parse_dates=["date"]
)
stock = stock.set_index("date").reindex(calendar["date"]).reset_index()
```

第一行的 `with_name` 将同一目录下的文件名从 `prices.csv` 换成 `calendar.csv`。这里读取独立保存的市场日历，不从某只股票有行情的日期反推整个市场日历。`set_index("date")` 把日期设为对齐标签，`reindex(calendar["date"])` 按日历重排并补出缺失的日期位置，`reset_index()` 再把日期还原为普通列。

这段代码只补出“应该存在的行位置”，没有补价格。若某个应有交易日缺少股票行情，就保留 `NaN` 以便调查，不能让该日期从表中消失。直接对不完整的股票记录做 `shift(1)` 时，前一条记录可能已经是几天前，计算结果便不再符合一个交易区间收益的定义。

当前人工样本价格完整，因此对齐后 `SYN001` 仍是 1,827 行。真实停牌和数据缺失还需要明确的估值与状态规则，不能在本节用“都填前值”替代这些判断。模拟日历也仍是工作日日历，不因代码使用了 `calendar` 这个名字就变成真实 A 股交易日历。

## 核心计算与字段说明

```python
stock["prev_adj_close"] = stock["adj_close"].shift(1)
stock["daily_return"] = stock["adj_close"] / stock["prev_adj_close"] - 1
stock["daily_return_pct"] = stock["daily_return"] * 100
```

第一行把上一行的复权收盘价移到当前行。`shift(1)` 不是把价格向未来移动来预测，而是在当前这一行放置历史值。计算前提是已经筛选出同一只股票，按日期排序并对齐日历；缺少这些前提，`shift` 本身不会自动理解金融含义。

第二行按同一行的当前价和历史价计算小数收益。第三行只增加一个阅读用百分数列，原始 `daily_return` 继续保存小数。例如 `-0.0293633667` 与 `-2.93633667%` 表示同一个收益，但百分数列中的数字是 `-2.93633667`。若把百分数数字继续拿去做 `1+r`，就会把量级放大 100 倍。

新增字段的含义如下：

| 字段 | 含义 |
| --- | --- |
| `prev_adj_close` | 当前日历行所对应的上一模拟交易日复权收盘价 |
| `daily_return` | 简单日收益，按小数存储，参与后续计算 |
| `daily_return_pct` | `daily_return×100`，数字单位是百分数，仅供阅读 |
| `return_shortcut` | pandas 简写生成的对照收益列 |

## 实际前六行结果

下表来自当前固定人工输入，显示值经过适度舍入；计算仍使用文件中的浮点数。

| 日期 | 复权收盘价 | 前一复权收盘价 | 日收益小数 | 日收益百分数 |
| --- | ---: | ---: | ---: | ---: |
| 2019-01-01 | 26.3273052964 | NaN | NaN | NaN |
| 2019-01-02 | 25.5542469758 | 26.3273052964 | -0.0293633667 | -2.93633667% |
| 2019-01-03 | 26.4112607851 | 25.5542469758 | 0.0335370402 | 3.35370402% |
| 2019-01-04 | 26.4755988379 | 26.4112607851 | 0.0024360084 | 0.24360084% |
| 2019-01-07 | 26.0559654690 | 26.4755988379 | -0.0158498160 | -1.58498160% |
| 2019-01-08 | 26.4957081427 | 26.0559654690 | 0.0168768520 | 1.68768520% |

我用 2019-01-02 手算一遍：`25.5542469758 / 26.3273052964 - 1 ≈ -0.0293633667`。同一变动若仅看绝对价差，大约是 `-0.7730583206`；这是价格单位，不是收益率。报告中需要同时标清数值与单位，避免把两个量混为一谈。

第一行没有当前快照之前的收盘价，所以 `shift(1)` 产生 `NaN`，日收益也为 `NaN`。这不足以断言当天没有涨跌，更不能将其视为 -100% 或 0%。若另一份数据提供了更早的可靠收盘价，可以扩展输入再计算，但不能手工捏造起点。

## 与 pandas 简写交叉核对

```python
stock["return_shortcut"] = stock["adj_close"].pct_change(fill_method=None)
max_difference = (
    stock["daily_return"] - stock["return_shortcut"]
).abs().max()
```

`pct_change` 的名字含有百分比，但输出仍是小数变化率，不会自动乘 100。显式设置 `fill_method=None`，表示不自动填充缺失价格。相减后取绝对值，再取最大值，可以检查两种实现有没有不一致。

当前结果为最大差异 **0.0**，总记录 **1,827**，日收益缺失 **1**。因此当前样本有 1,826 个可计算的相邻价格区间。这个检查说明两种写法在当前输入上等价，不说明输入本身必然正确；若两个方法都使用了同一份错误排序，它们也可能一起算错。因此，前面的代码、日期与行情核对仍然必要。

## 缺失价格与收益区间

假设日历包含 A、B、C 三个相邻交易日，而 B 的价格缺失。对齐后 B 的当日收益未知，C 的前一价格也未知，所以涉及这两个区间的日收益都不能直接计算。若先删掉 B，再把 A 到 C 的变化放到 C 行，它可能是一个多日累计变化，不是一个日收益。

本节保留报表空值，不用 `fillna(0)` 消除，因为零收益是有含义的有效观测。把未知替换成零，会影响后续复利、波动率、因子和统计样本量。若真实停牌需要使用已知估值，应在数据工程与会计规则中明确建立这一处理，而不是借一个方便的 pandas 语句悄悄完成。

## 收益、损益与信息时点

价格变化用比例描述，账户损益用金额描述。只有在这段变化发生前持有一定数量，且忽略其他现金流时，才能由数量乘价格差得到损益；没有持仓时，市场价格上涨不会自动产生账户收益。相同日收益可以对应不同初始资金下的不同盈亏金额，所以本节不能直接回答“我赚了多少钱”。

相邻交易区间与自然时间长度也需要区分。表里从星期五到星期一可能隔了三个自然日，但在这个日历中仍是一个相邻观测区间。不能把周末补成两条零收益记录，否则平均收益、波动率与后续窗口数量都会改变。真实日历和人工日历的差别，同样会影响这些统计口径。

当前行使用的收盘价，要到当日收盘后才完整可得。它适合描述已经结束的价格变化，不能支持在同一收盘前提前使用这份完整收益。后面构建因子时，需要明确哪些历史收益属于当时已知信息；构建未来标签时，则另外确定标签起点，防止把已经发生的变化混入“未来可预测收益”。

一天收益也不能直接乘以一年交易日数量，作为实际全年收益。真实累计收益需要连乘，年化指标又需要清楚的样本长度和约定。把不同问题留在相应步骤处理，可以避免一列数字被反复换名，却没有保持同一个定义。

## 常见错误与验收

| 错误操作 | 原因 |
| --- | --- |
| 对全表直接 `shift(1)` | 相邻行可能属于不同股票，会产生没有金融含义的价格比 |
| 使用自然日减一天寻找前值 | 周末和节假日使“前一天”不等于“上一交易日” |
| 把分母写成当天价格 | 衡量基准发生改变，不再是本节简单收益定义 |
| 把涨跌额当收益率 | 价格差有价格单位，收益率是相对变化 |
| 把 `NaN` 补成 0 | 未知变化被伪装成确定的零收益 |
| 忽略 `pct_change` 返回小数 | 百分数展示与后续计算会被混用 |

本步验收包含：单股、时间顺序、独立日历三个前提明确；第一行收益保留缺失；手算一行能对上；展开公式与简写差异为零。本步没有图表产物，主要证据是能够逐行回算的价格与收益表。

下一步进入 [第 03 步](../03_cumulative_returns/README.md)：把一连串日收益转成复利财富。后续继续保留“小数计算、百分数展示”的约定，并将首日未知收益与人为设定的净值基准分开理解。

## 完整可运行实现

上面的片段用于解释思路；下面各单元按顺序执行完整脚本。先定位仓库根目录，并显式设置脚本路径，使 `__file__` 在 Notebook 中也有明确含义。

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "pyproject.toml").exists()
                     and (candidate / "examples/01_read_prices.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("我需要从本项目目录或它的子目录打开 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
__file__ = str(PROJECT_ROOT / 'examples/02_daily_returns.py')
sys.argv = [__file__]


### 完整实现 · 单元 1

In [2]:
"""第二课：用复权收盘价计算一只模拟股票的简单日收益率。"""

from pathlib import Path
import pandas as pd

# 沿用上一课的读取、选股和排序。
project_dir = Path(__file__).resolve().parents[1]
data_path = project_dir / "runs" / "first_demo_20261107" / "input" / "prices.csv"
prices = pd.read_csv(data_path, dtype={"symbol": "string"}, parse_dates=["date"])
symbol = "SYN001"
stock = prices.loc[prices["symbol"] == symbol].copy()
stock = stock.sort_values("date").reset_index(drop=True)

# 对齐市场日历：缺失的交易日保留空价格，不能跳过缺口或自动补价。
calendar = pd.read_csv(data_path.with_name("calendar.csv"), parse_dates=["date"])
stock = stock.set_index("date").reindex(calendar["date"]).reset_index()

# 本课核心：今天的复权收盘价 / 上一个交易日的复权收盘价 - 1。
stock["prev_adj_close"] = stock["adj_close"].shift(1)
stock["daily_return"] = stock["adj_close"] / stock["prev_adj_close"] - 1

# 单独增加一列百分数，便于阅读；daily_return 仍保存原始小数。
stock["daily_return_pct"] = stock["daily_return"] * 100
columns = ["date", "adj_close", "prev_adj_close", "daily_return", "daily_return_pct"]
print("模拟股票：", symbol)
print("daily_return_pct 的单位为 %；NaN 表示缺失，不能当作零收益。")
print(stock[columns].head(6).to_string(index=False))

# pandas 的简写；fill_method=None 明确不自动补缺失价格。
stock["return_shortcut"] = stock["adj_close"].pct_change(fill_method=None)
max_difference = (stock["daily_return"] - stock["return_shortcut"]).abs().max()
print("\n展开公式与 pct_change 的最大差异：", max_difference)
print("行情记录数：", len(stock))
print("日收益缺失数：", int(stock["daily_return"].isna().sum()))

模拟股票： SYN001
daily_return_pct 的单位为 %；NaN 表示缺失，不能当作零收益。
      date  adj_close  prev_adj_close  daily_return  daily_return_pct
2019-01-01  26.327305             NaN           NaN               NaN
2019-01-02  25.554247       26.327305     -0.029363         -2.936337
2019-01-03  26.411261       25.554247      0.033537          3.353704
2019-01-04  26.475599       26.411261      0.002436          0.243601
2019-01-07  26.055965       26.475599     -0.015850         -1.584982
2019-01-08  26.495708       26.055965      0.016877          1.687685

展开公式与 pct_change 的最大差异： 0.0
行情记录数： 1827
日收益缺失数： 1


## 运行复核

以上实现已在干净内核中按顺序执行；执行状态记录在仓库的 [Notebook 校验记录](../../publication/notebook_validation.json)。代码中的数据身份、费用和成交假设仍然适用，运行成功只验证对应实现。